# Train a segmentation model on a mapcv dataset

A minimal, end-to-end PyTorch loop: load the [quickstart](../quickstart/) dataset with the
reusable [`MapcvDataset`](../scripts/torch_dataset.py) class, train a tiny U-Net for a couple
of epochs and report the mean intersection-over-union (mIoU) on the held-out test split.

The goal is to show the plumbing, not to reach a good score: 52 training patches and two epochs
give a rough model in about a minute on a laptop CPU. Raise `EPOCHS` (and the size of the
region in `mapcv.yaml`) for something useful.

**This notebook needs PyTorch**, which mapcv itself does not depend on. Install the CPU build
(or follow [pytorch.org](https://pytorch.org/get-started/locally/) for GPU builds):

```bash
pip install torch torchvision matplotlib
```

Only `torch` is required here; `torchvision` is needed if you swap in one of its ready-made
segmentation models (see the end of the notebook).

Generate the dataset first: `cd examples/quickstart && mapcv generate mapcv.yaml`.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
except ImportError as exc:
    raise ImportError("This notebook needs PyTorch: pip install torch torchvision") from exc

# MapcvDataset lives in examples/scripts/torch_dataset.py; copy it into your own project.
sys.path.insert(0, str(Path("../scripts").resolve()))
from torch_dataset import MapcvDataset

DATASET = Path("../quickstart/dataset")
EPOCHS = 2  # raise to 20+ for a useful model
BATCH_SIZE = 8

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} on {device}")

## 1. Data

`MapcvDataset(root, split)` reads the patch list from `splits/<split>.txt` and returns dicts
with a `float32 (C, H, W)` image and an `int64 (H, W)` mask. We add a transform that

* replaces NaN (NoData in Sentinel-2 patches) with 0,
* normalises each channel with the training set's mean and standard deviation,
* and, for training only, randomly rotates and flips image and mask together.

In [ ]:
raw_train = MapcvDataset(DATASET, "train")
stack = np.stack([np.nan_to_num(raw_train[i]["image"]) for i in range(len(raw_train))])
mean = stack.mean(axis=(0, 2, 3))
std = stack.std(axis=(0, 2, 3)) + 1e-6
del stack
print("channel mean", mean.round(3), "std", std.round(3))


def make_transform(train):
    rng = np.random.default_rng(0)

    def transform(sample):
        image, mask = np.nan_to_num(sample["image"]), sample["mask"]
        if train:
            k = int(rng.integers(4))
            image, mask = np.rot90(image, k, axes=(1, 2)), np.rot90(mask, k)
            if rng.random() < 0.5:
                image, mask = image[:, :, ::-1], mask[:, ::-1]
        image = (image - mean[:, None, None]) / std[:, None, None]
        return {
            **sample,
            "image": np.ascontiguousarray(image, dtype=np.float32),
            "mask": np.ascontiguousarray(mask),
        }

    return transform


datasets = {
    split: MapcvDataset(DATASET, split, transform=make_transform(train=split == "train"))
    for split in ("train", "val", "test")
}
loaders = {
    split: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=split == "train")
    for split, ds in datasets.items()
}
IGNORE_INDEX = 255  # mapcv's labels.ignore_index
num_classes = datasets["train"].num_classes
in_channels = datasets["train"][0]["image"].shape[0]
class_names = [datasets["train"].class_names.get(i, str(i)) for i in range(num_classes)]
print({split: len(ds) for split, ds in datasets.items()}, "classes:", class_names)

## 2. Model

A three-level U-Net with 16 base channels (~0.5 M parameters). It works for any number of
input bands, so the same code trains on the 4-band Sentinel-2 example.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class TinyUNet(nn.Module):
    def __init__(self, in_channels, num_classes, width=16):
        super().__init__()
        w = width
        self.pool = nn.MaxPool2d(2)
        self.enc1 = conv_block(in_channels, w)
        self.enc2 = conv_block(w, 2 * w)
        self.enc3 = conv_block(2 * w, 4 * w)
        self.bottleneck = conv_block(4 * w, 8 * w)
        self.up3 = nn.ConvTranspose2d(8 * w, 4 * w, 2, stride=2)
        self.dec3 = conv_block(8 * w, 4 * w)
        self.up2 = nn.ConvTranspose2d(4 * w, 2 * w, 2, stride=2)
        self.dec2 = conv_block(4 * w, 2 * w)
        self.up1 = nn.ConvTranspose2d(2 * w, w, 2, stride=2)
        self.dec1 = conv_block(2 * w, w)
        self.head = nn.Conv2d(w, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


model = TinyUNet(in_channels, num_classes).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

## 3. Train

Plain cross-entropy and Adam. After each epoch we report the validation mIoU: the IoU of
every class (background included), averaged. IoU is computed from a confusion matrix
accumulated over all pixels of the split. Pixels mapcv marks as having no imagery
(mask value 255, `labels.ignore_index`) are skipped by both the loss and the metric.

In [ ]:
@torch.no_grad()
def iou_per_class(model, loader):
    model.eval()
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    for batch in loader:
        pred = model(batch["image"].to(device)).argmax(dim=1).cpu()
        mask = batch["mask"]
        keep = mask != IGNORE_INDEX  # pixels without imagery are not scored
        index = mask[keep] * num_classes + pred[keep]
        confusion += torch.bincount(index, minlength=num_classes**2).reshape(
            num_classes, num_classes
        )
    intersection = confusion.diag().double()
    union = confusion.sum(0).double() + confusion.sum(1).double() - intersection
    return torch.where(union > 0, intersection / union, torch.nan)


optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)  # skip no-imagery pixels

for epoch in range(1, EPOCHS + 1):
    model.train()
    started, total_loss = time.time(), 0.0
    for batch in loaders["train"]:
        optimizer.zero_grad()
        loss = loss_fn(model(batch["image"].to(device)), batch["mask"].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch["image"])
    val_iou = iou_per_class(model, loaders["val"])
    print(
        f"epoch {epoch}: loss {total_loss / len(datasets['train']):.3f}, "
        f"val mIoU {val_iou.nanmean():.3f} ({time.time() - started:.0f}s)"
    )

## 4. Evaluate on the test split

The test patches come from spatial blocks the model never saw, so this is an honest
estimate of how it does on new ground in the same area.

In [ ]:
test_iou = iou_per_class(model, loaders["test"])
for name, iou in zip(class_names, test_iou.tolist()):
    print(f"{name:>12}: IoU {iou:.3f}")
print(f"{'test mIoU':>12}: {test_iou.nanmean():.3f}")

In [ ]:
def to_rgb(image):
    rgb = np.moveaxis(np.nan_to_num(image[:3]), 0, -1)  # first three bands
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)


model.eval()
raw_test = MapcvDataset(DATASET, "test")
n = min(4, len(raw_test))
fig, axes = plt.subplots(3, n, figsize=(3 * n, 9), squeeze=False)
for col in range(n):
    sample = datasets["test"][col]
    with torch.no_grad():
        pred = model(torch.from_numpy(sample["image"])[None].to(device)).argmax(1)[0].cpu()
    axes[0, col].imshow(to_rgb(raw_test[col]["image"]))
    axes[1, col].imshow(
        sample["mask"], vmin=0, vmax=num_classes - 1, cmap="tab10", interpolation="nearest"
    )
    axes[2, col].imshow(pred, vmin=0, vmax=num_classes - 1, cmap="tab10", interpolation="nearest")
for row, title in enumerate(["image", "ground truth", "prediction"]):
    axes[row, 0].set_ylabel(title)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Next

* Train longer (`EPOCHS = 30`) and on a bigger region; add a weighted loss if a class is rare.
* Swap the model for a pretrained one, e.g.
  `torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(num_classes=num_classes)`
  (its forward returns a dict; use `model(x)["out"]`).
* Try semi-supervised training with the `splits/10/labeled.txt` and `unlabeled.txt` lists:
  `MapcvDataset(DATASET, "10/labeled")`.
* Run the same notebook on the [Sentinel-2 example](../sentinel2-landcover/) by changing
  `DATASET`: the dataset class and model adapt to 4 float32 bands and 5 classes.